# ロジスティック回帰(演習)

解説資料は `research-handbook/machine-learning/classification.md`。
このnotebookは**直接編集せず**、自分の卒研repoにコピーして使うこと(手順は `technical-handbook/colab/use-github-repo.md`)。

「---- ここから課題 ----」の区間を埋めながら上から順に実行する。
解答付き版は `research-handbook/notebooks/logistic_regression_solution.ipynb` にあるが、まず自力で取り組むこと。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def make_blobs2(n=100, seed=0):
    rng = np.random.default_rng(seed)
    X0 = rng.normal([-1.0, -0.5], 0.8, (n // 2, 2))
    X1 = rng.normal([ 1.0,  0.8], 0.8, (n - n // 2, 2))
    X = np.vstack([X0, X1])
    t = np.concatenate([np.zeros(n // 2), np.ones(n - n // 2)])
    return X, t

X_train, t_train = make_blobs2(200, seed=0)
X_test,  t_test  = make_blobs2(200, seed=1)
Phi_train = np.hstack([np.ones((len(X_train), 1)), X_train])   # バイアス項を追加
Phi_test  = np.hstack([np.ones((len(X_test), 1)),  X_test])

## 課題1: シグモイド関数

$$\sigma(a) = \frac{1}{1 + e^{-a}}$$

素朴に実装すると $a$ が大きな負の値のとき `exp(-a)` がオーバーフローします。符号で場合分けして安定に実装します。

In [ ]:
def sigmoid(a):
    """課題1: シグモイド関数。オーバーフローしない実装にすること"""
    # ---- ここから課題1 ----
    # ヒント: a >= 0 のときは 1/(1+exp(-a))、a < 0 のときは exp(a)/(1+exp(a))
    # と場合分けすると exp の引数が常に非正になり安全


    # ---- ここまで ----

print(sigmoid(np.array([-800.0, -1.0, 0.0, 1.0, 800.0])))  # 0, 0.269, 0.5, 0.731, 1

## 課題2: 交差エントロピーの勾配

`classification.md` で導出したとおり、勾配は驚くほど簡単な形になります:

$$\nabla E(w) = \sum_n (y_n - t_n)\, \phi_n = \Phi^\top (y - t)$$

「予測と正解のズレ × 入力」という形は、banditのQ値更新・REINFORCEと同じ「予測誤差」の構造です。実装したら数値微分で検算します。

In [ ]:
def cross_entropy(w, Phi, t):
    """交差エントロピー損失 E(w) = -Σ [t ln y + (1-t) ln(1-y)]"""
    y = sigmoid(Phi @ w)
    eps = 1e-12
    return -np.sum(t * np.log(y + eps) + (1 - t) * np.log(1 - y + eps))

def grad_cross_entropy(w, Phi, t):
    """課題2: 勾配 ∇E = Φ^T (y - t)。classification.md の導出参照"""
    # ---- ここから課題2 ----
    # 完成したら下の数値微分チェックで検算すること
    return None
    # ---- ここまで ----

# 数値微分と比較して検算(この習慣はREINFORCE・ヤコビアンと同じ)
w0 = np.array([0.1, -0.2, 0.3])
g = grad_cross_entropy(w0, Phi_train, t_train)
eps = 1e-6
g_num = np.zeros_like(w0)
for j in range(len(w0)):
    e = np.zeros_like(w0); e[j] = eps
    g_num[j] = (cross_entropy(w0 + e, Phi_train, t_train) - cross_entropy(w0 - e, Phi_train, t_train)) / (2 * eps)
print("解析勾配と数値勾配の最大誤差:", np.abs(g - g_num).max())

## 課題3: 勾配降下による学習

損失が単調に減ること、テストaccuracyが高いことを確認し、決定境界 $w_0 + w_1 x_1 + w_2 x_2 = 0$ を描きます。

In [ ]:
def train_logistic(Phi, t, lr=0.01, n_iters=500):
    """課題3: 勾配降下による学習"""
    w = np.zeros(Phi.shape[1])
    history = []
    for i in range(n_iters):
        # ---- ここから課題3 ----
        # 勾配降下の1ステップ(最小化なので勾配を「引く」)
        pass
        # ---- ここまで ----
        history.append(cross_entropy(w, Phi, t))
    return w, history

w, history = train_logistic(Phi_train, t_train)
acc_train = np.mean((sigmoid(Phi_train @ w) > 0.5) == t_train)
acc_test  = np.mean((sigmoid(Phi_test  @ w) > 0.5) == t_test)
print(f"train accuracy = {acc_train:.3f}, test accuracy = {acc_test:.3f}")

plt.figure(figsize=(9, 3))
plt.subplot(1, 2, 1)
plt.plot(history); plt.xlabel("iteration"); plt.ylabel("cross entropy")
plt.subplot(1, 2, 2)
plt.scatter(*X_train.T, c=t_train, cmap="coolwarm", s=15)
xs = np.linspace(-3.5, 3.5, 2)
plt.plot(xs, -(w[0] + w[1] * xs) / w[2], "k-", label="decision boundary")  # w0+w1x+w2y=0
plt.legend(); plt.tight_layout(); plt.show()

## 確率的な出力

ロジスティック回帰は分類ラベルだけでなく**確率**を返します(SVMとの違い。`svm-kernel.md`)。境界から離れるほど確率が0/1に近づくことを確認します。

In [ ]:
# 決定境界から離れるほど確率が0/1に近づく(確率が出るのがSVMとの違い)
grid = np.array([[1, x1, x2] for x1 in np.linspace(-3, 3, 7) for x2 in [0.0]])
for row in grid:
    print(f"x = ({row[1]:+.1f}, {row[2]:.1f}) → p(C1|x) = {sigmoid(row @ w):.3f}")

## 発展課題

1. 学習率を10倍・100倍にすると損失曲線はどうなるか(`../reinforcement-learning/hyperparameter.md` の症状表と対応づける)
2. 2クラスの中心を近づけて(重なりを増やして)、accuracyと確率の出方がどう変わるか
3. L2正則化項 $\frac{\lambda}{2}\|w\|^2$ を損失と勾配に追加せよ(完全分離データで $\|w\|$ が発散する問題への対処。`classification.md`)

## まとめ

- ロジスティック回帰 = シグモイド + 交差エントロピー + 勾配法。勾配は $\Phi^\top(y-t)$
- ソフトマックス回帰に拡張すると、RLのソフトマックス方策・GAILの識別器と同じ数式になる
- 確率を返せることが決定理論(閾値調整)につながる(`model-evaluation.md`)